# [Chapter 4 — Force from Infection (Infected Viewpoint), §4.1-4.4] The Force from Infection $\alpha = c_I \beta P_S$

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 4 — Force from Infection (Infected Viewpoint)
**Considerations developed:** 4 (force of vs from), 8 (parameter estimation), 9 (practical fitting)
**Estimated runtime:** ≤ 1 minute

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
Develops the force from infection $\alpha = c_I \beta P_S$ from the infected viewpoint and explicitly contrasts the two estimators $\hat\alpha = J/I$ and $\hat\lambda = J/S$ under uncertainty in $S$. Demonstrates the structural-immunity advantage.

## What you should already know
Chapter 2A notebook on the susceptible viewpoint.


## Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import matplotlib.pyplot as plt
from shared import set_seed_chapter_02, book_style, BOOK_COLORS, integrate_sir_i
from shared.parameters import baseline_chapter_07

set_seed_chapter_02()
book_style()


## The infected viewpoint

An infectious individual makes $c_I$ contacts per day. The fraction of those contacts that are with susceptible individuals is $P_S = S/N$. Each susceptible-contact carries probability $\beta$ of producing a new infection. So an infectious individual creates new infections at rate:

$$\alpha = c_I \beta P_S = c_I \beta \frac{S}{N}$$

Total incidence is $J = \alpha I$. This is the **force from infection** — the per-infectious-person rate of producing new infections.

Why does this viewpoint matter? Because **the data observe the infectious person**, not the susceptible person. Surveillance counts cases (people who became infectious). It does not observe the susceptible population directly. Therefore, $\alpha$ is identifiable from data using $\hat\alpha = J/I$, while $\lambda$ requires assumptions about $S^*$.

In [ ]:
# The two estimators side-by-side
params = baseline_chapter_07()
result = integrate_sir_i(params)
t = result['t']
S = result['S']
I = result['I']
N = params['N']

# True alpha and lambda over time
alpha_true = params['c_I'] * params['beta'] * (S / N)
lambda_true = params['c_S'] * params['beta'] * (I / N)

# Incidence J (true)
J = alpha_true * I

# Two estimators (assume we observe J and I perfectly):
alpha_hat = J / I
lambda_hat_assume_N = J / N         # assumes S = N (typical wrong assumption)
lambda_hat_assume_S = J / S         # uses true S (typically unavailable)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(t, alpha_true, color=BOOK_COLORS['alpha_hat'], lw=2, label='$\\alpha(t)$ true')
ax.plot(t, alpha_hat, color=BOOK_COLORS['alpha_hat'], lw=1, ls='--', alpha=0.7, label='$\\hat\\alpha = J/I$')
ax.set_xlabel('Time (days)')
ax.set_ylabel('$\\alpha$ (per day)')
ax.set_title('Infected viewpoint: $\\hat\\alpha = J/I$')
ax.legend()

ax = axes[1]
ax.plot(t, lambda_true, color=BOOK_COLORS['lambda_hat'], lw=2, label='$\\lambda(t)$ true')
ax.plot(t, lambda_hat_assume_N, color=BOOK_COLORS['lambda_hat'], lw=1, ls='--', alpha=0.7, label='$\\hat\\lambda = J/N$ (assuming $S=N$)')
ax.set_xlabel('Time (days)')
ax.set_ylabel('$\\lambda$ (per day)')
ax.set_title('Susceptible viewpoint: $\\hat\\lambda = J/S^*$')
ax.legend()

plt.tight_layout()
plt.show()


## The structural-immunity advantage

Now we ask: how do the two estimators behave when $S^*$ is uncertain?

If we sweep the assumed $S^*$ across a plausible range, $\hat\alpha = J/I$ does not change at all (it doesn't reference $S^*$). But $\hat\lambda = J/S^*$ varies inversely with $S^*$ — a 30% error in $S^*$ produces a 30% error in $\hat\lambda$ and therefore in the resulting $R_0$ estimate.

This is the **structural-immunity** property: $\hat\alpha$ inherits zero sensitivity to assumed $N^*$, while $\hat\lambda$ inherits unit sensitivity.

In [ ]:
# Sensitivity sweep
S_star_range = np.linspace(0.5, 1.5, 21) * params['N']

# At a fixed time point during the outbreak
idx = np.argmax(I)  # at peak
J_peak = J[idx]
I_peak = I[idx]
S_peak_true = S[idx]

# alpha_hat doesn't reference S_star — it's invariant
alpha_hats = np.full_like(S_star_range, J_peak / I_peak)

# lambda_hat varies inversely with S_star
lambda_hats = J_peak / S_star_range

# Express as R_0 estimates (alpha_hat * tau_R, lambda_hat * tau_R / (I/N))
R0_from_alpha = alpha_hats * params['tau_R']
R0_from_lambda = lambda_hats * params['tau_R'] / (I_peak / params['N'])

true_R0 = params['c_I'] * params['beta'] * params['tau_R']

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(S_star_range / params['N'], R0_from_alpha, color=BOOK_COLORS['alpha_hat'], lw=2, label='$\\hat{R}_0$ from $\\hat\\alpha$ (infected viewpoint)')
ax.plot(S_star_range / params['N'], R0_from_lambda, color=BOOK_COLORS['lambda_hat'], lw=2, label='$\\hat{R}_0$ from $\\hat\\lambda$ (susceptible viewpoint)')
ax.axhline(true_R0, color=BOOK_COLORS['neutral'], ls='--', lw=1, alpha=0.6, label=f'True $R_0 = {true_R0:.2f}$')
ax.axvline(S_peak_true / params['N'], color=BOOK_COLORS['neutral'], ls=':', lw=1, alpha=0.5, label=f'True $S/N = {S_peak_true/params["N"]:.2f}$')
ax.set_xlabel('Assumed $S^* / N$')
ax.set_ylabel('Estimated $R_0$')
ax.set_title('Structural immunity: $\\hat\\alpha$ is flat, $\\hat\\lambda$ has unit sensitivity')
ax.legend()
plt.tight_layout()
plt.show()

print('Sensitivity index of $\\hat{R}_0$ from $\\hat\\alpha$ to $S^*$: 0  (flat line)')
print('Sensitivity index of $\\hat{R}_0$ from $\\hat\\lambda$ to $S^*$: -1  (inversely proportional)')


## What's next

This is the book's **central methodological contribution**. The infected viewpoint's structural-immunity advantage is what makes the framework's case-study comparisons (Chapter 17 COVID, Chapter 18 dengue, Chapter 19 HIV) tractable: in each case, the infected viewpoint absorbs the largest source of uncertainty.

The full theoretical development continues in:
- **Chapter 8 §sec:central-comparison** — formal proof and tornado-plot demonstration
- **Chapter 10** — sensitivity-analysis formalization
- **Chapter 20** — cross-pathogen demonstration